# RHOB Quickstart

Reward Hacking Onset Benchmark — evaluate a detector against a matched-proxy family in a single Colab runtime, no local install required.

This notebook:
1. Clones RHOB and installs dependencies
2. Evaluates a built-in detector on one family
3. Shows how to plug in your own detector
4. Runs the full access-level comparison (L0 vs L1 vs L2 vs L3)

Runtime: ~5 minutes end-to-end.

In [ ]:
!git clone --depth 1 https://github.com/Aarav500/rhob.git
%cd rhob
!pip install -q -e ".[dev]"

## 1. Evaluate a built-in detector

In [ ]:
from rhob.v3.benchmark import Benchmark
from rhob.detectors import RewardThresholdDetector

detector = RewardThresholdDetector()
results = Benchmark.evaluate(detector, families=["gridworld_camping"], n_seeds=10)
print(f"Overall AUROC: {results.overall_auroc:.3f}")

This is an L0 (reward-only) detector on a matched-proxy family, so ~0.50
(chance) is the *correct* result, not a bug — the whole point of RHOB is that
the proxy alone can't tell hacking from legitimate behavior.

## 2. Compare across the access-level hierarchy

In [ ]:
from rhob.detectors import (
    RewardThresholdDetector,
    StateDivergenceDetector,
    BehavioralThresholdDetector,
    PerfectFeatureOracleDetector,
)

detectors = {
    "L0 Reward Threshold": RewardThresholdDetector(),
    "L1 State Divergence": StateDivergenceDetector(),
    "L2 Behavioral Threshold": BehavioralThresholdDetector(),
    "L3 Perfect Feature Oracle": PerfectFeatureOracleDetector(),
}

for label, det in detectors.items():
    r = Benchmark.evaluate(det, families=["gridworld_camping"], n_seeds=10)
    print(f"{label:28s} AUROC={r.overall_auroc:.3f}")

## 3. Write and evaluate your own detector

In [ ]:
from rhob.detectors.posthoc import PosthocDetector, RunData
import numpy as np

class MyDetector(PosthocDetector):
    @property
    def access_level(self) -> str:
        return "L2"

    @property
    def name(self) -> str:
        return "My Detector"

    def classify(self, run: RunData) -> float:
        if run.behav_trace is None or len(run.behav_trace) < 2:
            return 0.5
        late = run.behav_trace[-100:]
        return float(1.0 / (1.0 + np.exp(-late.mean())))

    def detect_onset(self, run: RunData) -> int:
        trace = run.behav_trace
        if trace is None:
            return -1
        for t in range(10, len(trace)):
            if abs(trace[t]) > 0.5:
                return t
        return -1

results = Benchmark.evaluate(MyDetector(), families=["gridworld_camping"], n_seeds=10)
print(f"My detector AUROC: {results.overall_auroc:.3f}")

## Next steps

- [Detector Tutorial](https://github.com/Aarav500/rhob/blob/main/docs/TUTORIAL_DETECTOR.md) — full workflow including cross-family transfer
- [Environment Tutorial](https://github.com/Aarav500/rhob/blob/main/docs/TUTORIAL_ENVIRONMENT.md) — add a new hacking-mechanism family
- [CONTRIBUTING.md](https://github.com/Aarav500/rhob/blob/main/CONTRIBUTING.md) — submit your detector or family to RHOB